# 📐 Page Layout Segmentation — SAM on Medieval Manuscripts

Segmentation de structure de page (text/margin/illustration) à l'aide de SAM (Segment Anything Model).

**Pipeline:** Image page → SAM (régions) → Kraken BLLA (lignes) → OCR

In [ ]:
%%capture
!pip install segment-anything opencv-python-headless
!pip install numpy==1.26.4 --quiet
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load SAM
sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b_01ec64.pth')
sam = sam.to(device)
mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=1000,
)
print('✅ SAM loaded')

In [ ]:
# Load a manuscript page from CREMMA
!git clone https://github.com/HTR-United/cremma-medieval data/cremma 2>/dev/null || echo 'Already cloned'

# Find page images
cremma_dir = Path('data/cremma/data')
page_images = []
for ms_dir in sorted(cremma_dir.iterdir()):
    if not ms_dir.is_dir():
        continue
    for jpg in sorted(ms_dir.glob('*.jpg'))[:1]:
        page_images.append(str(jpg))

print(f'Found {len(page_images)} pages')
print(f'Using: {page_images[0]}')

## 1. SAM Automatic Segmentation

In [ ]:
# Run SAM on a manuscript page
img_path = page_images[0]
image = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f'Image size: {image_rgb.shape}')
print('Running SAM segmentation...')
masks = mask_generator.generate(image_rgb)
print(f'✅ {len(masks)} regions detected')

# Sort by area (largest first)
masks = sorted(masks, key=lambda x: x['area'], reverse=True)

In [ ]:
# Visualize segmentation
def show_masks(image, masks, title='SAM Segmentation'):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Original
    axes[0].imshow(image)
    axes[0].set_title('Image originale', fontsize=12)
    axes[0].axis('off')
    
    # With masks
    axes[1].imshow(image)
    for i, mask in enumerate(masks[:20]):  # Top 20 regions
        m = mask['segmentation']
        color = np.random.random(3)
        overlay = np.zeros_like(image, dtype=np.float32)
        overlay[m] = color
        axes[1].imshow(overlay, alpha=0.4)
    axes[1].set_title(f'SAM: {len(masks)} régions détectées', fontsize=12)
    axes[1].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('article/fig_sam_segmentation.png', dpi=300, bbox_inches='tight')
    plt.show()

show_masks(image_rgb, masks, 
           f'Segmentation SAM — {Path(img_path).stem}')

## 2. Region Classification (Text vs Non-Text)

In [ ]:
# Classify regions as text or non-text based on aspect ratio and density
def classify_region(mask, image_gray):
    """Classify a SAM mask as text, margin, or illustration."""
    m = mask['segmentation']
    bbox = mask['bbox']  # x, y, w, h
    area = mask['area']
    
    # Get bounding box aspect ratio
    x, y, w, h = bbox
    aspect_ratio = w / max(h, 1)
    
    # Calculate ink density (dark pixels in region)
    region_pixels = image_gray[m]
    ink_density = np.mean(region_pixels < 128)  # % of dark pixels
    
    # Heuristic classification
    img_h, img_w = image_gray.shape
    relative_area = area / (img_h * img_w)
    
    if relative_area > 0.3 and ink_density > 0.05 and ink_density < 0.5:
        return 'MainZone'  # Main text block
    elif relative_area < 0.05 and (x < img_w * 0.1 or x + w > img_w * 0.9):
        return 'MarginZone'  # Marginal annotation
    elif ink_density > 0.5:
        return 'DropCapitalZone'  # Decorated initial
    elif ink_density < 0.02:
        return 'Background'
    else:
        return 'TextZone'

# Classify all masks
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
classified = []
for mask in masks:
    label = classify_region(mask, image_gray)
    classified.append({'mask': mask, 'label': label})

# Count
from collections import Counter
labels = Counter(r['label'] for r in classified)
print('Regions classified:')
for label, count in labels.most_common():
    print(f'  {label}: {count}')

In [ ]:
# Visualize classified regions
fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(image_rgb)

colors = {
    'MainZone': (0, 1, 0),      # Green
    'TextZone': (0, 0.7, 0),    # Dark green
    'MarginZone': (1, 0.5, 0),  # Orange
    'DropCapitalZone': (1, 0, 0),  # Red
    'Background': (0.5, 0.5, 0.5),  # Gray
}

for r in classified:
    if r['label'] == 'Background':
        continue
    m = r['mask']['segmentation']
    color = colors.get(r['label'], (0, 0, 1))
    overlay = np.zeros((*image_rgb.shape[:2], 4), dtype=np.float32)
    overlay[m] = (*color, 0.3)
    ax.imshow(overlay)

# Legend
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items() if l != 'Background']
ax.legend(handles=legend_patches, loc='upper right', fontsize=10)
ax.set_title('Classification des régions (SAM + heuristiques)', fontsize=13)
ax.axis('off')

plt.tight_layout()
plt.savefig('article/fig_region_classification.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure saved')

## 3. Export PAGE XML

In [ ]:
import xml.etree.ElementTree as ET
from datetime import datetime

def export_page_xml(image_path, classified_regions, output_path):
    """Export SAM regions to PAGE XML format."""
    img = Image.open(image_path)
    w, h = img.size
    
    root = ET.Element('PcGts', {
        'xmlns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15'
    })
    metadata = ET.SubElement(root, 'Metadata')
    ET.SubElement(metadata, 'Creator').text = 'htr-medieval-french (SAM)'
    ET.SubElement(metadata, 'Created').text = datetime.utcnow().isoformat()
    
    page = ET.SubElement(root, 'Page', {
        'imageFilename': str(image_path),
        'imageWidth': str(w),
        'imageHeight': str(h)
    })
    
    for i, region in enumerate(classified_regions):
        if region['label'] == 'Background':
            continue
        
        mask = region['mask']
        bbox = mask['bbox']
        x, y, bw, bh = [int(v) for v in bbox]
        
        region_el = ET.SubElement(page, 'TextRegion', {
            'id': f'r{i}',
            'type': region['label']
        })
        coords = f'{x},{y} {x+bw},{y} {x+bw},{y+bh} {x},{y+bh}'
        ET.SubElement(region_el, 'Coords', {'points': coords})
    
    tree = ET.ElementTree(root)
    ET.indent(tree, space='  ')
    tree.write(output_path, encoding='utf-8', xml_declaration=True)
    return output_path

# Export
Path('segmentations').mkdir(exist_ok=True)
xml_path = export_page_xml(img_path, classified, 'segmentations/page_sam.xml')
print(f'✅ PAGE XML exported: {xml_path}')

# Show content
with open(xml_path) as f:
    print(f.read()[:2000])

## 4. Comparison: SAM vs Kraken BLLA

In [ ]:
# Run on multiple pages and compute statistics
results = []
for img_path in page_images[:5]:
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    masks = mask_generator.generate(image_rgb)
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    classified = []
    for mask in masks:
        label = classify_region(mask, image_gray)
        classified.append({'mask': mask, 'label': label})
    
    n_text = sum(1 for r in classified if r['label'] in ('MainZone', 'TextZone'))
    n_margin = sum(1 for r in classified if r['label'] == 'MarginZone')
    n_total = len(masks)
    
    results.append({
        'page': Path(img_path).stem,
        'total_regions': n_total,
        'text_regions': n_text,
        'margin_regions': n_margin,
    })
    print(f'  {Path(img_path).stem}: {n_total} regions ({n_text} text, {n_margin} margin)')

print(f'\n✅ Segmentation completed on {len(results)} pages')
print(f'   Average regions per page: {np.mean([r["total_regions"] for r in results]):.0f}')

## 5. Résumé

In [ ]:
print('=' * 60)
print('📐 RÉSUMÉ — Segmentation de page')
print('=' * 60)
print(f'  Modèle: SAM (Segment Anything) — vit_b')
print(f'  Pages traitées: {len(results)}')
print(f'  Régions moyennes/page: {np.mean([r["total_regions"] for r in results]):.0f}')
print(f'  Export: PAGE XML (compatible eScriptorium)')
print(f'  Pipeline: SAM (régions) → Kraken BLLA (lignes) → OCR')
print('=' * 60)
print()
print('Figures sauvegardées:')
print('  article/fig_sam_segmentation.png')
print('  article/fig_region_classification.png')